In [8]:
import yfinance as yf
import numpy as np
from scipy.optimize import minimize
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf
import statsmodels.api as sm
import seaborn as sns
from tqdm import tqdm

In [2]:
def clean_cov_rmt(cov_matrix, T, method='avg'):
    """
    Cleans a covariance matrix using Random Matrix Theory.

    Parameters:
    -----------
    cov_matrix : np.ndarray or pd.DataFrame
        The raw sample covariance matrix.
    T : int
        The number of observations (lookback window size) used to generate the matrix.
    method : str
        The replacement method used for eigenvalues that low below the expected maximum.
        Either avg to replace with average eigenvalue or zero to replace with zero.
    """
    cov_matrix = np.asarray(cov_matrix)
    N = cov_matrix.shape[0]

    # Decompose into Correlation Matrix
    std_devs = np.sqrt(np.diag(cov_matrix))

    # Prevent division by zero if an asset has zero variance
    std_devs = np.where(std_devs == 0, 1e-8, std_devs)

    # Extract correlation matrix
    corr_matrix = cov_matrix / np.outer(std_devs, std_devs)

    # Extract eigenvalues and vectors
    eigenvalues, eigenvectors = np.linalg.eigh(corr_matrix)

    # Sort eigenvalues and vectors in descending order
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    Q = N / T

    # The theoretical upper bound for noise eigenvalues by Marcenko-Pastur
    lambda_max = (1.0 + np.sqrt(Q))**2

    # Find all eigenvalues that fall below the noise threshold
    noise_idx = np.where(eigenvalues < lambda_max)[0]

    if len(noise_idx) > 0:
        # Replacement method for
        if method == 'avg':
            # Average the noise eigenvalues
            noise_avg = np.mean(eigenvalues[noise_idx])
            eigenvalues[noise_idx] = noise_avg
        elif method == 'zero':
            eigenvalues[noise_idx] = 0

    cleaned_corr = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T

    np.fill_diagonal(cleaned_corr, 1.0)

    cleaned_cov = cleaned_corr * np.outer(std_devs, std_devs)

    return cleaned_cov

In [3]:
def compute_mvp_weights(cov_matrix):
    """Global Minimum Variance Portfolio (MVP) - Minimizes portfolio variance."""
    cov_matrix = np.asarray(cov_matrix)
    num_assets = len(cov_matrix)

    def portfolio_variance(weights):
        return np.dot(weights.T, np.dot(cov_matrix, weights))

    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    initial_weights = np.array([1.0 / num_assets] * num_assets)

    result = minimize(fun=portfolio_variance, x0=initial_weights, method='SLSQP',
                      bounds=bounds, constraints=constraints, tol=1e-6)
    return result.x

def compute_mdp_weights(cov_matrix):
    """Maximum Diversification Portfolio (MDP) - Maximizes Diversification Ratio."""
    cov_matrix = np.asarray(cov_matrix)
    num_assets = len(cov_matrix)
    asset_vols = np.sqrt(np.diag(cov_matrix))

    def negative_diversification_ratio(weights):
        weighted_vol = np.dot(weights, asset_vols)
        port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        if port_vol == 0:
            return 0
        return -weighted_vol / port_vol

    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    initial_weights = np.array([1.0 / num_assets] * num_assets)

    result = minimize(fun=negative_diversification_ratio, x0=initial_weights, method='SLSQP',
                      bounds=bounds, constraints=constraints, tol=1e-6)
    return result.x

def compute_erc_weights(cov_matrix):
    """Equal Risk Contribution (ERC / Risk Parity) - Equalizes risk assets' spend."""
    cov_matrix = np.asarray(cov_matrix)
    num_assets = len(cov_matrix)

    def erc_objective(weights):
        port_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
        if port_variance <= 0: return 0

        # Marginal Risk Contribution (MRC) and Total Risk Contribution (TRC)
        marginal_risk = np.dot(cov_matrix, weights)
        risk_contributions = weights * marginal_risk / port_variance

        # Target contribution for each asset is exactly 1 / N
        target = 1.0 / num_assets
        # Penalty: sum of squared deviations from the target allocation
        return np.sum((risk_contributions - target) ** 2)

    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    initial_weights = np.array([1.0 / num_assets] * num_assets)

    result = minimize(fun=erc_objective, x0=initial_weights, method='SLSQP',
                      bounds=bounds, constraints=constraints, tol=1e-6)
    return result.x

In [4]:
def run_rolling_backtest(returns: pd.DataFrame, window_size: int, weight_func, use_rmt: bool = False, use_ds: bool = False, method='avg'):
    """
    Executes a rolling out-of-sample backtest to evaluate portfolio stability and volatility.

    Parameters:
    -----------
    returns : pd.DataFrame
        Historical daily returns (rows = dates, columns = asset tickers).
    window_size : int
        The lookback period T (e.g., 60, 120, 252).
    weight_func : function
        The allocation function (e.g., compute_mvp_weights, compute_erc_weights).
    use_rmt : bool
        If True, applies RMT cleaning to the covariance matrix before allocation.
    method : str
        Method used to replace eigenvalues. Either avg or zero.

    Returns:
    --------
    annual_volatility : float
        The annualized standard deviation of the out-of-sample returns.
    daily_turnover : float
        The average daily percentage of the portfolio traded to maintain target weights.
    oos_returns : pd.Series
        The daily out-of-sample portfolio returns.
    """
    num_days, num_assets = returns.shape

    out_of_sample_returns = np.zeros(num_days)
    weights_history = np.zeros((num_days, num_assets))

    # Rolling Loop
    for i in tqdm(range(window_size, num_days), desc='Rolling Loop'):
        # 1. Train Window: Day t-window_size to t-1
        train_data = returns.iloc[i - window_size : i]

        # Calculate standard sample covariance matrix
        cov_matrix = train_data.cov().values


        # Apply rmt if flagged
        if use_rmt:
            cov_matrix = clean_cov_rmt(cov_matrix, T=window_size, method=method)

        # Allocate weights based on weight_func
        try:
            w = weight_func(cov_matrix)
        except Exception:
            # Fallback to equal weight if SciPy's optimizer crashes on a highly singular Raw matrix
            print("Fallback to equal weight")
            w = np.ones(num_assets) / num_assets

        weights_history[i] = w

        # Apply the trained weights to day's actual returns
        today_returns = returns.iloc[i].values
        out_of_sample_returns[i] = np.dot(w, today_returns)

    # Slice off the initial training period
    oos_returns = out_of_sample_returns[window_size:]
    weights_history = weights_history[window_size:]

    # Calculate Stability Metric: Time-Series of Daily Turnover
    weight_changes = np.abs(np.diff(weights_history, axis=0))
    daily_turnover_series = np.sum(weight_changes, axis=1)

    # Format out-of-sample returns as a Series with dates
    oos_series = pd.Series(oos_returns, index=returns.index[window_size:])

    # Format turnover as a Series with dates
    turnover_series = pd.Series(daily_turnover_series, index=returns.index[window_size + 1:])

    # Calculate the single average number
    avg_turnover = turnover_series.mean()
    annual_volatility = oos_series.std() * np.sqrt(252)

    return annual_volatility, avg_turnover, oos_series, turnover_series

In [ ]:
sp500_meta = pd.read_csv("SP500_Wiki.csv")
sp500_cleaned_meta = sp500_meta.drop_duplicates(subset=['CIK'], keep='first')

# Limit to assets in the Financials sector
sp500_financials = sp500_cleaned_meta.loc[sp500_cleaned_meta['GICS Sector'] == "Financials"]
unique_tickers = sp500_financials['Symbol'].tolist()
financial_tickers = [t.strip().replace('.', '-') for t in unique_tickers]

data = yf.download(financial_tickers, start='2016-01-01', end='2026-01-01', progress=True)

adj_close = data['Close']

daily_returns = adj_close.pct_change()

financial_returns = daily_returns.iloc[1:].dropna(axis=1, how='any')

In [6]:
def calculate_advanced_metrics(raw_returns, rmt_returns, risk_free_rate=0.0):
    """
    Calculates Sharpe, Sortino, Maximum Drawdown (MDD), and Levene's test.
    Assumes daily percentage returns.
    """
    # Annualized Sharpe Ratios
    raw_sharpe = (raw_returns.mean() - risk_free_rate) / raw_returns.std() * np.sqrt(252)
    rmt_sharpe = (rmt_returns.mean() - risk_free_rate) / rmt_returns.std() * np.sqrt(252)

    # Annualized Sortino Ratios
    raw_downside = raw_returns[raw_returns < 0]
    rmt_downside = rmt_returns[rmt_returns < 0]

    raw_sortino = (raw_returns.mean() - risk_free_rate) / raw_downside.std() * np.sqrt(252)
    rmt_sortino = (rmt_returns.mean() - risk_free_rate) / rmt_downside.std() * np.sqrt(252)

    # Maximum Drawdown (MDD)
    def get_mdd(returns_series):
        # Convert daily returns to a cumulative wealth index
        cumulative_wealth = (1 + returns_series).cumprod()
        # Track the highest peak achieved so far
        running_max = cumulative_wealth.cummax()
        # Calculate percentage drop from the highest peak
        drawdown = (cumulative_wealth - running_max) / running_max
        return drawdown.min()

    raw_mdd = get_mdd(raw_returns)
    rmt_mdd = get_mdd(rmt_returns)

    # Levene's Test for Equality of Variances
    # Note that the observations are not independent. The assumptions are not met
    stat, p_value = stats.levene(raw_returns, rmt_returns)

    # --- Print Outputs ---
    print("--- Risk-Adjusted Performance ---")
    print(f"Raw Sharpe:  {raw_sharpe:.4f} | RMT Sharpe:  {rmt_sharpe:.4f}")
    print(f"Raw Sortino: {raw_sortino:.4f} | RMT Sortino: {rmt_sortino:.4f}")

    print("\n--- Tail Risk (Worst-Case Loss) ---")
    print(f"Raw MDD:    {raw_mdd * 100:.2f}% | RMT MDD:    {rmt_mdd * 100:.2f}%")

    print("\n--- Statistical Significance (Levene's Test) ---")
    print(f"Test Statistic: {stat:.4f}")
    print(f"P-Value:        {p_value:.6f}")

    if p_value < 0.05:
        print("Result: SIGNIFICANT. RMT mathematically altered the portfolio variance.")
    else:
        print("Result: NOT SIGNIFICANT. The variance difference is indistinguishable from noise.")

In [ ]:
rmt_methods = [(False, '', 'raw'), (True, 'avg', 'rmt_avg'), (True, 'zero', 'rmt_zero')]
rolling_window_sizes = [60, 120, 252]
allocation_methods = [compute_mvp_weights, compute_mdp_weights, compute_erc_weights]
names = ['MV', 'MD', 'ERC']

for rmt_method in rmt_methods:
    results = {}

    # Calculate and Store
    for size in rolling_window_sizes:
        for name, allocation_method in zip(names, allocation_methods):
            print(f"Calculating {rmt_method[2]} {name} for {size}-day window...")

            volatility, turnover, returns, turnover_series = run_rolling_backtest(
                returns=financial_returns,
                window_size=size,
                weight_func=allocation_method,
                use_rmt=rmt_method[0],
                method=rmt_method[1]
            )

            results[(name, size)] = [volatility, turnover, returns, turnover_series]

    # Export and Print
    for item in results:
        name = item[0]
        size = item[1]
        volatility = results[item][0]
        turnover = results[item][1]
        returns_series = results[item][2]
        turnover_series = results[item][3]

        # Export Returns
        csv_filename_returns = f"{name}_{size}_{rmt_method[2]}_returns.csv"
        returns_series.to_csv(csv_filename_returns, index=True, header=['OOS_Return'])

        # Export Turnover
        csv_filename_turnover = f"{name}_{size}_{rmt_method[2]}_turnover.csv"
        turnover_series.to_csv(csv_filename_turnover, index=True, header=['Daily_Turnover'])

        print(f"({name}, {size}): Volatility = {volatility:.4f}, Turnover = {turnover:.4f}")

In [ ]:
def cumulative_turnover(raw_turnover, rmt_turnover, rmt_zero_turnover, window_size, model_name):
    # Fix the X-axis
    raw_turnover.index = pd.to_datetime(raw_turnover.index)
    rmt_turnover.index = pd.to_datetime(rmt_turnover.index)
    rmt_zero_turnover.index = pd.to_datetime(rmt_zero_turnover.index)

    # Calculate Cumulative Turnover
    cum_raw = raw_turnover.cumsum() * 100
    cum_rmt = rmt_turnover.cumsum() * 100
    cum_zero = rmt_zero_turnover.cumsum() * 100

    plt.figure(figsize=(12, 6))

    # Plot the cumulative lines
    plt.plot(cum_raw.index, cum_raw.values, label=f'Raw {model_name} Turnover', color='red', linewidth=2)
    plt.plot(cum_rmt.index, cum_rmt.values, label=f'RMT-Cleaned w/ Average {model_name} Turnover', color='blue', linewidth=2)
    plt.plot(cum_zero.index, cum_zero.values, label=f'RMT-Cleaned w/ Zero {model_name} Turnover', color='green', linewidth=2)

    # Highlight the COVID-19 Crash
    plt.axvspan(pd.to_datetime('2020-02-15'), pd.to_datetime('2020-05-15'),
                color='grey', alpha=0.3, label='COVID-19 Crash')

    # Formatting
    plt.title(f'Cumulative Portfolio Turnover: {model_name} (T={window_size} days)', fontsize=14)
    plt.ylabel('Cumulative Turnover (%)', fontsize=12)
    plt.xlabel('Date', fontsize=12)
    plt.legend(loc='upper left', fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

rolling_window_sizes = [60, 120, 252]
names = ['MV', 'MD', 'ERC']
for name in names:
    for size in rolling_window_sizes:
        turnover_raw_series = pd.read_csv(f"{name}_{size}_raw_turnover.csv", index_col=0)["Daily_Turnover"]
        turnover_rmt_series = pd.read_csv(f"{name}_{size}_rmt_avg_turnover.csv", index_col=0)["Daily_Turnover"]
        turnover_zero_series = pd.read_csv(f"{name}_{size}_rmt_zero_turnover.csv", index_col=0)["Daily_Turnover"]
        cumulative_turnover(turnover_raw_series, turnover_rmt_series, turnover_zero_series, size, name)

In [ ]:
rolling_window_sizes = [60, 120, 252]
names = ['MV', 'MD', 'ERC']
alternative_methods = ['rmt_avg', 'rmt_zero']

for alternative in alternative_methods:
    for name in names:
        for size in rolling_window_sizes:
            fig, ax = plt.subplots(figsize=(10, 5))
            raw_turnover = pd.read_csv(f"{name}_{size}_raw_turnover.csv", index_col=0)["Daily_Turnover"]
            rmt_turnover = pd.read_csv(f"{name}_{size}_{alternative}_turnover.csv", index_col=0)["Daily_Turnover"]
            daily_difference = raw_turnover - rmt_turnover
            plot_acf(daily_difference, lags=20, title=f"Autocorrelation for {name} ({size}) w/ {alternative}", ax=ax)
            ax.set_xlabel('Lag (in Days)')
            ax.set_xlim([0, 20])
            # plt.savefig(f"autocorrelation_{model}_{time}.pdf", bbox_inches="tight")
            plt.show()

In [ ]:
rolling_window_sizes = [60, 120, 252]
names = ['MV', 'MD', 'ERC']
alternative_methods = ['rmt_avg', 'rmt_zero']

for alternative in alternative_methods:
    for name in names:
        for size in rolling_window_sizes:
            raw_turnover = pd.read_csv(f"{name}_{size}_raw_turnover.csv", index_col=0)["Daily_Turnover"]
            rmt_turnover = pd.read_csv(f"{name}_{size}_{alternative}_turnover.csv", index_col=0)["Daily_Turnover"]
            daily_difference = raw_turnover - rmt_turnover

            Y = daily_difference
            X = np.ones(len(Y))

            # Fit OLS with Newey-West HAC standard errors
            num_lags = int(4 * (len(Y) / 100) ** (2 / 9))
            model = sm.OLS(Y, X)
            results = model.fit(cov_type='HAC', cov_kwds={'maxlags': num_lags})

            # Extract the coefficient and two-sided p-value
            beta_hat = results.params[0]
            p_two_sided = results.pvalues[0]

            # Calculate the one-sided p-value for Ha: Beta > 0
            if beta_hat > 0:
                p_one_sided = p_two_sided / 2.0
            else:
                p_one_sided = 1.0 - (p_two_sided / 2.0)

            # Output
            print(f"===== {name} Sumnmary for {alternative} vs raw =====")
            print(results.summary())
            print("-" * 50)
            print(f"Coefficient (Beta): {beta_hat:.6f}")
            print(f"Two-sided P-value (Ha: Beta != 0): {p_two_sided:.6f}")
            print(f"One-sided P-value (Ha: Beta > 0):  {p_one_sided:.6f}")

In [ ]:
raw_data = yf.download('^GSPC', start='2016-01-01', end='2026-01-01', progress=True)['Close']

# 5. Compute Market Beta and Annualized Market Alpha relative to S&P 500
def compute_capm_metrics(asset_series, market_series, rf_annual=0.04, trading_days=252):
    # Daily risk-free rate
    rf_daily = (1 + rf_annual) ** (1 / trading_days) - 1

    # Excess returns
    r_p = asset_series - rf_daily
    r_m = market_series - rf_daily

    # OLS Regression via Covariance
    cov_matrix = np.cov(r_p, r_m)
    beta_market = cov_matrix[0, 1] / cov_matrix[1, 1]

    # Alpha calculation
    alpha_daily = r_p.mean() - (beta_market * r_m.mean())
    alpha_annual = alpha_daily * trading_days

    # R-squared (explanatory power of S&P 500 for this portfolio)
    correlation = np.corrcoef(r_p, r_m)[0, 1]
    r_squared = correlation ** 2

    return beta_market, alpha_annual, r_squared

capm_data = []
rolling_window_sizes = [60, 120, 252]
names = ['MV', 'MD', 'ERC']
rmt_methods = ['raw', 'rmt_avg', 'rmt_zero']
print("=== PORTFOLIO METRICS VS. S&P 500 ===")
for name in names:
  for size in rolling_window_sizes:
    for alternative in rmt_methods:
      portfolio_returns = pd.read_csv(f"{name}_{size}_{alternative}_returns.csv")
      index1 = portfolio_returns['Date'][0]
      market_returns = raw_data.pct_change().iloc[1:].dropna(axis=0, how='any')['^GSPC'].loc[index1:]
      portfolio_returns = portfolio_returns['OOS_Return']
      beta_m, alpha_m, r2 = compute_capm_metrics(portfolio_returns, market_returns)

      capm_data.append({"Strategy": name, 'Window':size, "Method":alternative, 'Beta': beta_m, 'Alpha':alpha_m*100, 'R2':r2})

In [ ]:
df = pd.DataFrame(capm_data)

# Set global style
sns.set_theme(style="whitegrid")

# Heatmap of RMT Impact
# Pivot data to calculate rmt - raw
alternative_methods = ['rmt_avg', 'rmt_zero']
for method in alternative_methods:
    pivot_df = df.pivot_table(index=['Strategy', 'Window'], columns='Method', values='Alpha')
    pivot_df['RMT_Impact'] = pivot_df[f'{method}'] - pivot_df['raw']
    heatmap_data = pivot_df['RMT_Impact'].unstack(level=1)

    plt.figure(figsize=(8, 5))
    sns.heatmap(heatmap_data, annot=True, cmap="RdYlGn", center=0, fmt=".3f")
    if method == 'rmt_avg':
        plt.title("Alpha Delta: Impact of Average RMT-cleaning vs Raw")
    else:
        plt.title("Alpha Delta: Impact of Zero RMT-cleaning vs Raw")
    plt.ylabel("Strategy")
    plt.xlabel("Rolling Window")
    plt.show()